In [ ]:
!date

### generate a bins annotation file to filter

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Skin', 'Inflamed', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
bins_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/windows/500bp/GRCh38.500bp.bed'
bins = pd.read_csv(bins_path, sep='\t', header=None)
bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
bins.shape

In [ ]:
bins.head()

### add features now

In [ ]:
annot_df = pd.DataFrame(index=bins.index)
annot_df.head()

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
bins_bt = BedTool.from_dataframe(bins)

bins_pmds = bins_bt.intersect(pmds_bt, wa=True).to_dataframe()
bins_pmds.index = [f'{x}_{y}' for x,y in zip(bins_pmds['chrom'], bins_pmds['start'])]
bins_pmds.shape

In [ ]:
annot_df['is_pmd'] = 0
mask = annot_df.index.isin(bins_pmds.index)
annot_df.loc[mask, 'is_pmd'] = 1
annot_df['is_pmd'].value_counts()

### add chromhmm

In [ ]:
### old - uses only intersecting ATAC peaks 
# ### makes sense to do Start only features?
# # chromhmm.index = [f'{x}_{y}' for x,y in zip(chromhmm['seqnames'], chromhmm['start'])]
# chromhmm.shape

# %%time

In [ ]:
# %%time

# ## a little slow

In [ ]:
### use all vu states
vu_states_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc/bed/justin/vu.states.bed.gz'
vu_states = pd.read_csv(vu_states_path, sep='\t', header=None)
vu_states.shape

In [ ]:
vu_states.head()

In [ ]:
classes = vu_states[6].unique()
classes

In [ ]:
vu_states[6].value_counts()

In [ ]:
%%time

def get_annot(s):
    tmp_vu_states = vu_states[vu_states[6] == s]
    vu_states_bt = BedTool.from_dataframe(tmp_vu_states)
    bins_inter = bins_bt.intersect(vu_states_bt, wa=True).to_dataframe()
    bins_inter.index = [f'{x}_{y}' for x,y in zip(bins_inter['chrom'], bins_inter['start'])]
    mask = annot_df.index.isin(bins_inter.index)
    return mask

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(get_annot, classes), total=len(classes)))

In [ ]:
for i, annot in enumerate(classes):
    annot_df[f'is_{annot}'] = 0
    mask = results[i]
    annot_df.loc[mask, f'is_{annot}'] = 1

In [ ]:
annot_df.head()

### genes

In [ ]:
genes_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/gencode.v43.autosome.dedup.names.bed'
genes = pd.read_csv(genes_path, sep='\t', header=None)
genes.index = genes[3]
genes.shape

In [ ]:
genes.head()

In [ ]:
genes[0]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
degs = pd.read_csv(f'{indir}/csv/degs/justin_degs_v7.csv', sep='\t', index_col=0)
degs.shape

In [ ]:
start_degs = degs[degs['coarse_v7'] == 'Start'].index
start_degs = start_degs[~start_degs.duplicated()]
start_genes = genes.reindex(start_degs).dropna().sort_values(by=[0, 1])
start_genes[1] = start_genes[1].astype(int)
start_genes[2] = start_genes[2].astype(int)
start_genes.shape

In [ ]:
start_genes.head()

In [ ]:
genes_bt = BedTool.from_dataframe(genes)
bins_inter = bins_bt.intersect(genes_bt, wa=True).to_dataframe()
bins_inter.index = [f'{x}_{y}' for x,y in zip(bins_inter['chrom'], bins_inter['start'])]

annot_df[f'is_gene'] = 0
mask = annot_df.index.isin(bins_inter.index)
annot_df.loc[mask, f'is_gene'] = 1

### cpg islands

In [ ]:
cpg_islands_path = '/u/project/cluo/terencew/reference/hg38/annotations/cpgIslandExt.txt.gz'

cpg_islands = pd.read_csv(cpg_islands_path, sep='\t', header=None, index_col=0)
cpg_islands.shape

In [ ]:
cpg_islands_bt = BedTool.from_dataframe(cpg_islands)
bins_inter = bins_bt.intersect(cpg_islands_bt, wa=True).to_dataframe()
bins_inter.index = [f'{x}_{y}' for x,y in zip(bins_inter['chrom'], bins_inter['start'])]

annot_df[f'is_cpg_island'] = 0
mask = annot_df.index.isin(bins_inter.index)
annot_df.loc[mask, f'is_cpg_island'] = 1

In [ ]:
annot_df.to_csv(f'{projdir}/csv/variance/bins_500bp_annot.csv.gz', sep='\t')

In [ ]:
!date